In [ ]:
Initialize ActiveAgent (The one learning, requires gradients)
Initialize FrozenAgent (The opponent/transitioner, eval mode, no gradients)
Initialize Optimizer for ActiveAgent
Initialize Environment env

# Target Phase defines who is training (e.g., 2 for Combat, 0 for Shop)
TARGET_PHASE_ID = 2

For iteration = 1 to MAX_ITERATIONS:

    # ==========================================
    # PHASE 1: Data Collection (Rollout)
    # ==========================================
    Reset env, get initial state s and initial action_masks
    Initialize empty Buffer: states, masks, actions, rewards, log_probs, values, dones

    For step = 1 to ROLLOUT_STEPS:
        current_phase = s.run_token.phase

        If current_phase == TARGET_PHASE_ID:
            # ------------------------------------------
            # ACTIVE PHASE (Learning Agent takes the wheel)
            # ------------------------------------------
            # 1. Forward Pass & Masking
            sel_logits, exec_logits, value = ActiveAgent.forward(s)
            sel_logits[~action_masks.card_select] = -1e9
            exec_logits[~action_masks.execute] = -1e9

            # 2. Sample Multi-Discrete Action & Log Probs
            sel_dist, exec_dist = Categorical(sel_logits), Categorical(exec_logits)
            selections, execution = sel_dist.sample(), exec_dist.sample()
            log_prob = sel_dist.log_prob(selections).sum() + exec_dist.log_prob(execution)

            # 3. Step Environment
            action_dict = {"selection": selections, "execute": execution}
            next_s, reward, real_done, next_masks = env.step(action_dict)

            # 4. Phase Transition Logic (Virtual Done)
            # If the next state is a different phase, the active agent's episode is over.
            next_phase = next_s.run_token.phase
            virtual_done = real_done or (next_phase != TARGET_PHASE_ID)

            # 5. Store Trajectory Data in Buffer
            Buffer.store(s, action_masks, action_dict, reward, log_prob, value, virtual_done)

        Else:
            # ------------------------------------------
            # FROZEN PHASE (Just advance the environment)
            # ------------------------------------------
            # 1. Forward Pass & Masking (No Gradients)
            with no_grad():
                sel_logits, exec_logits, _ = FrozenAgent.forward(s)

            sel_logits[~action_masks.card_select] = -1e9
            exec_logits[~action_masks.execute] = -1e9

            # 2. Sample Action
            sel_dist, exec_dist = Categorical(sel_logits), Categorical(exec_logits)
            selections, execution = sel_dist.sample(), exec_dist.sample()

            # 3. Step Environment (Do NOT store data, do NOT track rewards)
            action_dict = {"selection": selections, "execute": execution}
            next_s, _, real_done, next_masks = env.step(action_dict)

        # Advance State
        s, action_masks = next_s, next_masks

        If real_done:
            s, action_masks = env.reset()

    # ==========================================
    # PHASE 2: Optimization (Only for ActiveAgent)
    # ==========================================
    # 1. Calculate GAE using the Buffer's virtual_dones
    # This prevents values from bleeding across phase transitions
    next_value = ActiveAgent.forward(s)[2]
    advantages = ComputeGAE(Buffer.rewards, Buffer.values, next_value, Buffer.virtual_dones)
    returns = advantages + Buffer.values
    advantages = (advantages - mean(advantages)) / (std(advantages) + 1e-8)

    # 2. PPO Epochs
    For epoch = 1 to PPO_EPOCHS:
        Shuffle Buffer data and split into MINIBATCHES

        For each minibatch in MINIBATCHES:
            # Standard PPO Math on the Active Agent
            curr_sel_logits, curr_exec_logits, curr_values = ActiveAgent.forward(minibatch.states)

            curr_sel_logits[~minibatch.masks.card_select] = -1e9
            curr_exec_logits[~minibatch.masks.execute] = -1e9

            curr_sel_dist = Categorical(logits=curr_sel_logits)
            curr_exec_dist = Categorical(logits=curr_exec_logits)

            curr_log_probs = curr_sel_dist.log_prob(minibatch.actions.selection).sum(dim=-1) + \
                             curr_exec_dist.log_prob(minibatch.actions.execute)
            entropy = curr_sel_dist.entropy().sum(dim=-1) + curr_exec_dist.entropy()

            # Loss Functions
            ratio = exp(curr_log_probs - minibatch.old_log_probs)
            surrogate1 = ratio * minibatch.advantages
            surrogate2 = clip(ratio, 1 - epsilon, 1 + epsilon) * minibatch.advantages
            actor_loss = -min(surrogate1, surrogate2).mean()

            critic_loss = MSE(curr_values, minibatch.returns).mean()
            total_loss = actor_loss + (c_value * critic_loss) - (c_entropy * entropy.mean())

            # Backpropagate
            Optimizer.zero_grad()
            total_loss.backward()
            clip_grad_norm_(ActiveAgent.parameters(), max_grad_norm)
            Optimizer.step()